In [ ]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import warpSPHCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import warpSPHCore as sph
from warpSPHCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from warpSPHIntegrators.integration import *
from warpSPHCore import *
from warpSPHPlotting import *

# This library
from warpSPH import *

# The case utilities that contain all the case setup functions for the various test cases
from warpSPH.caseUtils import *

In [ ]:
directories = []

directories.append(f'export/semiPeriodic/')
directories.append(f'export/dambreak/')
directories.append(f'export/fullyPeriodic/')
directories.append(f'export/openChannel/')
directories.append(f'export/boundedRandom/')
directories.append(f'export/boundedRandom_wObstacle/')
directories.append(f'export/periodicRandom/')
directories.append(f'export/periodicRandom_wObstacle/')

# for now just take the newest subdirectory in each directory
simulationDirectories = []
for directory in directories:
    subdirs = [f.path for f in os.scandir(directory) if f.is_dir()]
    if len(subdirs) > 0:
        subdirs.sort(key=lambda x: os.path.getmtime(x), reverse=True)
        simulationDirectories.append(subdirs[0])
print(f"Found {len(simulationDirectories)} simulation directories: {simulationDirectories}")

directory = simulationDirectories[2]

trajectoryFile = f'{directory}/trajectory.h5'
configFile = f'{directory}/config.json'

In [ ]:
import h5py
import json

loadedConfig = json.load(open(configFile, 'r'))

trajectory = h5py.File(trajectoryFile, 'r')
numStates = len(trajectory['states'])
print(f'Loaded trajectory with {numStates} states from {trajectoryFile}')

In [ ]:
exportInterval = 0.002
# dt = loadedConfig['config']['dt']
state_keys = list(trajectory['states'].keys())
state_keys = sorted(state_keys, key=lambda x: int(x.split('_')[1]))
dt = trajectory['states'][state_keys[1]].attrs['time'] - trajectory['states'][state_keys[0]].attrs['time']

exportRatio = exportInterval / dt
exportRatio_i = int(exportRatio)
print(f'Export ratio: {exportRatio} [{exportRatio_i}] (dt={dt}, exportInterval={exportInterval})')

folderName = os.path.basename(trajectoryFile.split('/')[-2])
print(f'Loaded simulation from {folderName} with {numStates} states, export ratio: {exportRatio} [{exportRatio_i}] (dt={dt}, exportInterval={exportInterval})')

outFilePath = f'compressed/{trajectory.attrs["caseName"]}_{folderName}.h5'
os.makedirs(os.path.dirname(outFilePath), exist_ok=True)

In [ ]:
outFile = h5py.File(outFilePath, 'w')

In [ ]:
for attr in trajectory.attrs:
    outFile.attrs[attr] = trajectory.attrs[attr]

In [ ]:
outFile.attrs['exportInterval'] = exportInterval

In [ ]:
removeGhost = True
kinds = trajectory['initialState']['kinds'][:]
mask = kinds != 2 if removeGhost else np.ones_like(kinds, dtype=bool)
mask = kinds == 0
print(f'Initial state: {np.sum(mask)} particles after removing ghost particles (removeGhost={removeGhost})')

In [ ]:
boundaryMask = kinds == 1
boundaryPositions = trajectory['initialState']['positions'][boundaryMask]
boundaryMasses = trajectory['initialState']['masses'][boundaryMask]
boundarySupports = trajectory['initialState']['supports'][boundaryMask]
boundaryUIDs = trajectory['initialState']['UIDs'][boundaryMask]
boundaryKinds = trajectory['initialState']['kinds'][boundaryMask]

outFile.create_dataset('boundaryPositions', data=boundaryPositions, dtype=np.float32)
outFile.create_dataset('boundaryMasses', data=boundaryMasses, dtype=np.float32)
outFile.create_dataset('boundarySupports', data=boundarySupports, dtype=np.float32)
outFile.create_dataset('boundaryUIDs', data=boundaryUIDs, dtype=np.int32)
outFile.create_dataset('boundaryKinds', data=boundaryKinds, dtype=np.int32)

if hasattr(trajectory['initialState'], 'ghostOffsets'):
    boundaryGhostOffsets = trajectory['initialState']['ghostOffsets'][boundaryMask]
    outFile.create_dataset('boundaryGhostOffsets', data=boundaryGhostOffsets, dtype=np.int32)
else:
    print('No ghostOffsets found in initialState, skipping boundaryGhostOffsets')

fluidMask = kinds == 0
fluidPositions = trajectory['initialState']['positions'][fluidMask]
fluidMasses = trajectory['initialState']['masses'][fluidMask]
fluidSupports = trajectory['initialState']['supports'][fluidMask]
fluidUIDs = trajectory['initialState']['UIDs'][fluidMask]
fluidKinds = trajectory['initialState']['kinds'][fluidMask]

outFile.create_dataset('fluidPositions', data=fluidPositions, dtype=np.float32)
outFile.create_dataset('fluidMasses', data=fluidMasses, dtype=np.float32)
outFile.create_dataset('fluidSupports', data=fluidSupports, dtype=np.float32)
outFile.create_dataset('fluidUIDs', data=fluidUIDs, dtype=np.int32)
outFile.create_dataset('fluidKinds', data=fluidKinds, dtype=np.int32)

In [ ]:
compressedPositions = []
compressedVelocities = []
compressedDensities = []

compressedPositions.append(trajectory['initialState']['positions'][fluidMask])
compressedVelocities.append(trajectory['initialState']['velocities'][fluidMask])
compressedDensities.append(trajectory['initialState']['densities'][fluidMask])

In [ ]:
times = []
times.append(trajectory['initialState'].attrs['time'])

for k, key in tqdm(enumerate(state_keys)):
    if (k+1) % exportRatio_i == 0 and k > 0:
        # compressedPositions.append(trajectory['states'][key]['positions'][:])
        # compressedVelocities.append(trajectory['states'][key]['velocities'][:])
        # compressedDensities.append(trajectory['states'][key]['densities'][:])
        times.append(trajectory['states'][key].attrs['time'])
times = np.array(times, dtype=np.float32)
print(f'Compressed {len(times)} states from {numStates} states, export ratio: {exportRatio} [{exportRatio_i}] (dt={dt}, exportInterval={exportInterval})')
print(f'Times shape: {times.shape}, dtype: {times.dtype}, min: {np.min(times)}, max: {np.max(times)}')
if 'times' not in outFile:
    outFile.create_dataset('times', data=times, dtype=np.float32)
else:
    del outFile['times']
    outFile.create_dataset('times', data=times, dtype=np.float32)
    # outFile['times'][:] = times
# del(times)

In [ ]:
totalPositionsShape = (len(times),) + compressedPositions[0].shape
print(f'Total positions shape: {totalPositionsShape}, dtype: {compressedPositions[0].dtype}')
totalPositionBytes = np.prod(totalPositionsShape) * compressedPositions[0].dtype.itemsize
print(f'Total positions bytes: {totalPositionBytes} ({totalPositionBytes/1024**2:.2f} MB)')

In [ ]:
compressedPositions = []
compressedPositions.append(trajectory['initialState']['positions'][fluidMask])

for k, key in tqdm(enumerate(state_keys)):
    if (k+1) % exportRatio_i == 0 and k > 0:
        compressedPositions.append(trajectory['states'][key]['positions'][fluidMask])
        # compressedVelocities.append(trajectory['states'][key]['velocities'][:])
        # compressedDensities.append(trajectory['states'][key]['densities'][:])
        # times.append(trajectory['states'][key].attrs['time'])
pos = np.array(compressedPositions, dtype=np.float32)
print(f'Compressed {len(compressedPositions)} states from {numStates} states, export ratio: {exportRatio} [{exportRatio_i}] (dt={dt}, exportInterval={exportInterval})')
print(f'Positions shape: {pos.shape}, dtype: {pos.dtype}, min: {np.min(pos)}, max: {np.max(pos)}')
if 'positions' not in outFile:
    outFile.create_dataset('positions', data=pos, dtype=np.float32)
else:
    outFile['positions'][:] = pos
del(pos)

In [ ]:
trajectory['states'][f'frame_{0:05d}'].attrs.keys()
trajectory['initialState'].attrs.keys()

In [ ]:
trajectory.keys()

In [ ]:
def load_state(stateIndex, file):
    frame_keys = list(file['states'].keys())
    frame_key = frame_keys[stateIndex] if 0 <= stateIndex < len(frame_keys) else frame_keys[0]

    state = file['states'][frame_key]
    positions = torch.tensor(state['positions'][:], dtype=torch.float32)
    velocities = torch.tensor(state['velocities'][:], dtype=torch.float32)
    densities = torch.tensor(state['densities'][:], dtype=torch.float32)
    masses = torch.tensor(file['initialState']['masses'][:], dtype=torch.float32)
    supports = torch.tensor(file['initialState']['supports'][:], dtype=torch.float32)
    kinds = torch.tensor(file['initialState']['kinds'][:], dtype=torch.int32)
    UIDs = torch.tensor(file['initialState']['UIDs'][:], dtype=torch.int64)

    return WeaklyCompressibleState(
        positions=positions,
        velocities=velocities,
        densities=densities,
        masses=masses,
        supports=supports,
        kinds=kinds,
        UIDs=UIDs,
        materials = torch.zeros_like(kinds, dtype=torch.int32),
        UIDcounter = UIDs.max().cpu().item() + 1
    ), state.attrs['time']


In [ ]:
print(trajectory.attrs.keys())

In [ ]:
caseName = trajectory.attrs['caseName']
timeLimit = trajectory.attrs['timeLimit']
# dt = trajectory['states']['frame_00001'].attrs['time'] - trajectory['states']['frame_00000'].attrs['time']
nx = trajectory.attrs['nx']
n_h = trajectory.attrs['n_h']
L = trajectory.attrs['L']
W = trajectory.attrs['W']
obstacleType = trajectory.attrs['obstacleType']
aoa = trajectory.attrs['aoa']
obstacleActive = trajectory.attrs['obstacleActive']

dt = loadedConfig['config']['dt']
fixedSoundSpeed = loadedConfig['schemeConfig']['fixedSoundSpeed']

device = torch.device('cpu')
dtype = torch.float32

# dx = loadedConfig['config']['dx']
# band = trajectory.attrs['band']
dim = loadedConfig['config']['dim']
domain = buildDomainDescription(L, dim, True, device, dtype)
domain.min = torch.tensor(loadedConfig['config']['domain']['min'], device=device, dtype=dtype)
domain.max = torch.tensor(loadedConfig['config']['domain']['max'], device=device, dtype=dtype)

In [ ]:
markerSize = 6
plotWidth = 28
plotHeight = 10

In [ ]:
stateIndex = 0
state, time = load_state(stateIndex, trajectory)

caseText = f'{caseName}'
timeText = f't = {time:.4g}/{timeLimit:.4g} | dt = {dt:.4g}'
particleText = f'particles = {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary | nx = {nx} | n_h = {n_h}'
domainText = f'L = {L}, W = {W}'
obstacleText = f'obstacle: {obstacleType}, aoa: {aoa}' if obstacleActive else 'no obstacle'
stateText = f'v_max = {state.velocities.max().cpu().item():.4g} (c0 = {fixedSoundSpeed:.4g}), rho_max = {state.densities.max().cpu().item():.4g}, rho_min = {state.densities.min().cpu().item():.4g}'
timingText = f'iter time: {0.00:.3f} ms'

titleString = f'{caseText} | {timeText} | {particleText} | {domainText} | {obstacleText} | {stateText} | {timingText}'

from ipywidgets import widgets

def update_frame(frame_index):
    global state, time
    state, time = load_state(frame_index, trajectory)
    plotter.updateQuantities({
        # "A": state.velocities,
        "A": state.UIDs
    }, newParticleState=state)
    caseText = f'{caseName}'
    timeText = f't = {time:.4g}/{timeLimit:.4g} | dt = {dt:.4g}'
    particleText = f'particles = {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary | nx = {nx} | n_h = {n_h}'
    domainText = f'L = {L}, W = {W}'
    obstacleText = f'obstacle: {obstacleType}, aoa: {aoa}' if obstacleActive else 'no obstacle'
    stateText = f'v_max = {state.velocities.max().cpu().item():.4g} (c0 = {fixedSoundSpeed:.4g}), rho_max = {state.densities.max().cpu().item():.4g}, rho_min = {state.densities.min().cpu().item():.4g}'
    timingText = f'iter time: {0.00:.3f} ms'

    titleString = f'{caseText} | {timeText} | {particleText} | {domainText} | {obstacleText} | {stateText} | {timingText}'

    plotter.updateTitle(titleString)


markerSize = 12
velocityPlot = PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            mapping = Mapping.L2Norm,
            plotTitle = "Particle Velocity Magnitude",
            plotTitleGap = 0.08,
            boundaryVisualization = VisualizeOptions.Visualize,
            # gridVisualization = GridVisualization(
            #     resolution = 1024,
            #     streamLines = True,
            # ),

            # vMin=1e-10,
            vMin = 0.0,
            vMax = fixedSoundSpeed * 0.1,
        )
densityPlot = PlottingOptions(
            colorMap = DivergingColorMap.RdBu,
            flipColorMap=True,
            markerSize = markerSize,
            midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "Particle Density",
            # vMin = 0.95,
            # vMax = 1.05,
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )
UIDPlot = PlottingOptions(
            colorMap = CyclicColorMap.twilight,
            # flipColorMap=True,
            markerSize = markerSize,
            # midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = f"Particle IDs ({caseName},  {len(state.positions[state.kinds == 0])} fluid + {len(state.positions[state.kinds == 1])} boundary particles)",
            boundaryVisualization = VisualizeOptions.Passive,
            # vMin = 0.95,
            # vMax = 1.05
            plotTitleGap = 0.08,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        )


In [ ]:

plotter = visualize(
    particleState = state,
    domain = domain,
    quantities = {
        # "A": state.velocities,
        "A": state.UIDs
    },
    plotOptions = {
        # "A": velocityPlot,
        "A": UIDPlot
    },
    figTitle = titleString,
    mosaic = 'A',
    figsize= (plotWidth, plotHeight),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

plotter.updateTitle(titleString)


frame_slider = widgets.IntSlider(
    value=500,
    min=0,
    max=numStates-1,
    step=1,
    description='Frame:',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

frame_slider.observe(lambda change: update_frame(change['new']), names='value')
# update_frame(frame_slider.value)

export_button = widgets.Button(
    description='Export Current Frame',
    button_style='success',
    tooltip='Export the current frame as an image',
    icon='download'
)
export_button.on_click(lambda b: plotter.export(f'{caseName}_frame_{frame_slider.value:05d}.png'))


display(frame_slider)
display(export_button)


In [ ]:
uMax = torch.linalg.norm(state.velocities, dim=1).max().cpu().item()
dx = state.masses[state.kinds == 0].mean().cpu().item() ** (1.0 / dim)  # approximate particle spacing based on mass and density
cfl = uMax * dt / dx
print(f"Max velocity: {uMax:.4g}, dx: {dx:.4g}, dt: {dt:.4g}, CFL number: {cfl:.4g}")

targetCFL = 1.0
maxDt = targetCFL * dx / uMax
print(f"Target CFL: {targetCFL:.4g}, Max dt for target CFL: {maxDt:.4g}")
dtRatio = dt / maxDt
print(f"dt ratio: {dtRatio:.4g} (dt / maxDt for target CFL)")
print(f'Maximum Coarse Graining Factor: {1/dtRatio:.4g} (1 / dtRatio for target CFL)')